# CAES mini-project: system design and simulation

Compressed Air Energy Storage (CAES) stores electricity by compressing air into a reservoir. During discharge, the air is heated and expanded through a turbine to produce electricity. The project compares how compression heat management and reservoir design affect performance.

## System variants

The six variants combine three architectures with two reservoir types:

| Architecture | Constant-volume reservoir | Constant-pressure reservoir |
|---|---|---|
| Single-stage compression and expansion; compression heat rejected | 1a | 1b |
| Single-stage with thermal energy storage (TES) | 2a | 2b |
| Two-stage compression with intercooling and two-stage expansion with reheat; no TES | 3a | 3b |

A constant-volume reservoir operates between 16 bar and a minimum pressure of 8 bar, so its pressure falls during discharge. A constant-pressure reservoir remains at 16 bar while its volume varies. Variants 2a and 2b store compression heat in a hot-water TES and reuse it to preheat discharge air before combustion. Variants 3a and 3b use equal pressure-ratio stages; intercooling heat is rejected, and fuel reheats the air before each turbine stage.

## Common specification

- Working fluid: real-gas air (CoolProp).
- Ambient conditions: 15°C and 1.013 bar.
- Maximum storage pressure: 16 bar; minimum pressure for constant-volume storage: 8 bar.
- Target output: 1 MW at the turbine/generator shaft for 4 hours (4 MWh per cycle); charge duration is also 4 hours.
- Compressor isentropic efficiency: 0.85; turbine isentropic efficiency: 0.88.
- Mechanical/generator efficiency: 0.98.
- Turbine inlet temperature: 550°C. Combustion is represented as heat addition to reach this temperature, without modelling combustion chemistry.
- Cooler and TES return pinches: 10 K. Air is cooled towards ambient temperature plus the cooler pinch before storage.
- The reservoir is treated as a uniform 0D store. Stored air is assumed to return to ambient temperature before discharge and remain near ambient during discharge.

## Performance and design tasks

Use the same two-step workflow for the selected variant:

1. **Design:** size the required components, heat exchangers, and storage volume for the common output target.
2. **Simulation:** model a complete charge/discharge cycle and evaluate performance.

The common round-trip metric values fuel at its electrical equivalent:

\[
\eta = \frac{W_{\mathrm{turbine}}}{W_{\mathrm{compressor}} + \eta_f Q_{\mathrm{combustion}}},
\qquad \eta_f = 0.40.
\]

Compare round-trip efficiency, storage volume, heat-exchanger areas, maximum temperatures, and the electrical/fuel input split. Include energy (and optionally exergy) balances, identify major losses, and quantify at least one proposed performance improvement. The final comparison should show how thermal storage, staging, and reservoir type affect these results.

Here are some sources i used to return to later (soso)

Return to Gas Turbines EENG4 S8 for compressor eq and details (soso)

Paper on which sile will the pinch temp be occuring based on the Cc and Ch (LMTD) method. https://sci-hub.fr/10.1016/B978-008042839-0/50008-2 (soso)

Also return to EENG3 S6 Lec 4 for counter flow details and eq (soso)

In [118]:
# --- imports ---
from pathlib import Path
import sys

for root in (Path.cwd(), *Path.cwd().parents):
    for package_root in (root, root / "caldis-0.3.0"):
        if (package_root / "caldis" / "__init__.py").is_file():
            sys.path.insert(0, str(package_root))
            break
    else:
        continue
    break

import numpy as np
import caldis
from CoolProp.CoolProp import PropsSI
from caldis.core.system import System
from caldis.components import FluidBoundary
from caldis.components import Compressor
from caldis.fluids import CoolPropBackend
from caldis.components import SizingHX
from caldis.components import CombustionChamber
from caldis.components import Turbine
from caldis.components.vessels.tank import TankConstP
from caldis.solvers import solve  


Initilize paramaters

Ambient Air --> Compressor --> Cooler --> Air Storage --> Combustor --> Turbine --> Exhuast

In [119]:
# --- system definition ---
#Inlet Compressor 
T0=15 +273.15 #k
P0=1.013e5
Eta_ISC=0.85

#Cooler (water)
Tpinch=10 #k


#Air Storage
Pmax=16e5
    #Volume is Variable probably use constraint to find volume based on mass flow rate and time of storage

#Combustion Chamber
Ttit=550 +273.15 #k
Eta_Fuel=0.4

#Turbine
Eta_IST=0.85
Eta_Mech=0.98

Paramters to assume

In [120]:
# A1: First we define a mass flow rate
Mdot_air=1 #kg/s
Mdot_water=1 #kg/s

# A2: We define a guess for the water outlet temperature
T_water_out_Guess = 100 + 273.15 #K This is an assumption cuz it must be less than 400 and more than T0+Tpinch

# A3: We define the backend for the fluids which is the film convection coefficients
# The heat exchanger is a counter flow heat exchanger with water on one side and air on the other side. 
# The heat transfer coefficients are based on the type of fluid and the flow conditions. Typical values are:
#   forced convection in gas (air): 50–250 W/m²/K
#   forced convection in liquid (water): 500–10000 W/m²/K
h_film_air   = 100.0    # W/m²/K  (air side)
h_film_water = 3000.0   # W/m²/K  (water side)

T_out_comp_guess = 400 + 273.15 #K This is an assumption cuz it must be less than Ttit and more than T0

Ratio_Area_cold_to_hot = 0.5 # This is an assumption, it can be changed to see the effect on the performance of the system


In [121]:
# --- Component setup ---
be_air   = CoolPropBackend("Air")
be_water = CoolPropBackend("Water")
be_fuel = CoolPropBackend("Methane")

h_tank = PropsSI('H', 'T', T0, 'P', Pmax, 'Air')

#Tank = TankConstP(name="Tank", backend=be_air, p=Pmax, h_init=h_tank, V_init=None, C_wall=None, T_wall_init=T0, UA_wall=None, Q_heater=0)

Source = FluidBoundary(name="Source", p=Pmax, T=T0, h=None, mdot=None, backend=be_air)

Comb= CombustionChamber(name="comb", backend_air=be_air, backend_fuel=be_fuel, LHV=50e5, eta_comb=1.0, dp_rel=0.04)
Fuel_in = FluidBoundary(name="Fuel_in", backend=be_fuel)

Turbine = Turbine(name="turb", backend=be_air, eta_is=Eta_IST, eta_mech=Eta_Mech)

Exhaust= FluidBoundary(name="Exhaust", p=P0, T=None, h=None, mdot=None, backend=be_air)


In [ ]:
# --- System wiring & start points ---
sys_ = System("Comp_Parts")
sys_.add(Source, Turbine, Comb, Fuel_in)

sys_.connect(Source.port, Comb.air_in)
sys_.connect(Fuel_in.port, Comb.fuel_in)

sys_.connect(Comb.gas_out, Turbine.inl)
sys_.connect(Turbine.out, Exhaust.port)


h_turbine_in = PropsSI('H', 'T', Comb.gas_out.T, 'P', Pmax, 'Air')
h_turbine_out = PropsSI('H', 'T', Turbine.out.T, 'P', Pmax, 'Air')

sys_.add_constraint(lambda: Comb.gas_out.T - 550 + 273, "max T")
#sys_.add_constraint(lambda: Turbine.inl.p - Pmax, "[p_comb]")
sys_.add_constraint(lambda: (Eta_Mech*Eta_IST*(Turbine.out.h-Turbine.inl.h)*(4*60*60))/(3.6*10^9))


sys_.set_start_port(Source.port, p=Pmax, T=T0)

sys_.set_start_port(Comb.air_in, p=Pmax, T=T0)
sys_.set_start_port(Comb.gas_out, p=Pmax, T=Ttit)
sys_.set_start_port(Turbine.inl, p=Pmax, T=Comb.gas_out.T)
sys_.set_start_port(Turbine.out, p=P0, T=250)
#sys_.set_start_port(Exhaust.inl, p=P0, T=Turbine.out.T)



sys_.init_starts()


sol = solve(sys_)

SyntaxError: '(' was never closed (3892686738.py, line 17)

In [123]:
#sys_.structure()
Comb.describe()


=== CombustionChamber('comb') ===
  ports (p [bar], mdot [kg/s], T [°C]):
    air_in     p=16.000  mdot=+0.01  h=410.7kJ/kg  T=15.00
    fuel_in    p=1.000  mdot=+0.01  h=876.7kJ/kg  T=10.00
    gas_out    p=16.000  mdot=+0.01  h=974.5kJ/kg  T=550.00
  outputs:
    Qdot_fuel        = 50000
    Qdot_released    = 50000
    AFR              = 1
    excess_air       = 0.0581395


{'variables': {},
 'ports': {'air_in': {'p': 1600000.0, 'mdot': 0.01, 'h': 410734.7295205181},
  'fuel_in': {'p': 100000.0, 'mdot': 0.01, 'h': 876735.6080682593},
  'gas_out': {'p': 1600000.0, 'mdot': 0.01, 'h': 974452.4547996501}},
 'outputs': {'Qdot_fuel': 50000.0,
  'Qdot_released': 50000.0,
  'AFR': 1.0,
  'excess_air': 0.05813953488372093}}